# 🎯 Tutorial 07: Skill Discovery y Composición## Aprendiendo Habilidades Reutilizables AutomáticamenteEn este tutorial completo aprenderás:- 🧩 Qué son skills, options, y primitivas de comportamiento- 🎯 DIAYN: Diversity is All You Need para skill discovery- 🏛️ Option-Critic: Aprendizaje end-to-end de opciones- 🔮 Successor Features para transfer de skills- 💻 Implementación completa de skill discovery- 🎼 Composición de skills para tareas complejas- 📊 Evaluación de diversidad y utilidad de skills- 🤖 Aplicaciones en robótica y juegos**Tiempo estimado**: 90-120 minutos---

## 📑 Table of Contents- [1 - Introduction to Skills and Options](#1)    - [1.1 - The Compositionality Problem](#1-1)    - [1.2 - What are Skills?](#1-2)    - [1.3 - Why Learn Skills Automatically?](#1-3)- [2 - Setup and Dependencies](#2)- [3 - Theoretical Background](#3)    - [3.1 - Options Framework](#3-1)    - [3.2 - DIAYN: Diversity-Driven Discovery](#3-2)    - [3.3 - Option-Critic Architecture](#3-3)    - [3.4 - Successor Features](#3-4)- [4 - Exercise 1 - Grid World Environment](#ex-1)- [5 - Exercise 2 - Skill Discriminator](#ex-2)- [6 - Exercise 3 - Skill Policy](#ex-3)- [7 - Exercise 4 - DIAYN Training Loop](#ex-4)- [8 - Evaluation: Skill Diversity](#8)- [9 - Visualization: Skill Behaviors](#9)- [10 - Experiment: Compositional Tasks](#10)- [11 - Comparison: Skills vs Flat Policy](#11)- [12 - Real-World Applications](#12)- [13 - Advanced Topics](#13)- [14 - Summary and Conclusions](#14)

<a name='1'></a>## 1 - Introduction to Skills and Options<a name='1-1'></a>### 1.1 - The Compositionality Problem**Observation**: Humans learn by composing simple skills into complex behaviors.**Examples:**| Complex Task | Decomposition ||--------------|--------------|| Make coffee | Walk to kitchen + Grab mug + Use coffee machine + Carry back || Play soccer | Run + Kick + Pass + Dribble + Position || Drive to work | Start car + Navigate streets + Park + Walk to building || Cook dinner | Chop vegetables + Heat pan + Sauté + Season + Plate |**The Problem with Flat Policies:**Traditional RL learns a monolithic policy: $\pi(a|s)$❌ **Challenges:**1. **Sample inefficiency**: Must relearn everything for each new task2. **No transfer**: Learning "walk to X" doesn't help with "walk to Y"3. **Exploration**: Random actions rarely accomplish anything meaningful4. **Interpretability**: Can't understand what the policy is doing5. **Scalability**: Complex tasks have enormous action spaces**Example: Robot Manipulation**```Flat policy: 1,000,000 training episodes per taskWith skills:  Learn skills: {grasp, release, push, pull} - 200,000 episodes total  Compose for new tasks: 10,000 episodes per taskTotal for 10 tasks: 300,000 vs 10,000,000 episodesSpeedup: 33x!```

<a name='1-2'></a>### 1.2 - What are Skills?**Skills** (also called **options** or **primitives**) are temporally-extended actions.**Formal Definition:**A skill/option is a tuple: $o = (I, \pi, \beta)$- $I \subseteq S$: **Initiation set** - states where skill can start- $\pi: S \times A \rightarrow [0,1]$: **Policy** - how the skill behaves- $\beta: S \rightarrow [0,1]$: **Termination** - probability of ending**Intuition:**- **Initiation**: "When can I use this skill?"- **Policy**: "What do I do while executing this skill?"- **Termination**: "When do I stop?"**Example: "Walk to Door" Skill**```pythonI = {all states where door is visible}π = {actions that move toward door}β = {high probability when at door, low otherwise}```**Hierarchical Decision Making:**```Meta-Controller:  Select which skill to use: ω ~ μ(s)Skill Policy:  Execute actions: a ~ π_ω(s)  Until termination: β_ω(s)Then repeat```**Benefits of Skills:**1. **Temporal abstraction**: One decision executes many actions2. **Structured exploration**: Skills guide exploration meaningfully3. **Transfer**: Skills learned in one task help in others4. **Sample efficiency**: Reuse skills rather than relearn5. **Interpretability**: "Use skill 3" is more meaningful than "action 47"**Types of Skills:**| Type | Description | Example ||------|-------------|---------|| Primitive | Single action | "Move left" || Reactive | Stateless policy | "Follow wall" || Goal-conditioned | Reach specific state | "Go to (x,y)" || Context-dependent | Behavior varies | "Grasp" (depends on object) |

<a name='1-3'></a>### 1.3 - Why Learn Skills Automatically?**Manual skill design is hard:**❌ Requires domain expertise❌ Not scalable to complex environments❌ May miss useful skills❌ Doesn't adapt to environment**Automatic skill discovery** learns skills from experience:✅ **No human expertise** needed✅ **Discovers unexpected** useful skills✅ **Adapts** to environment specifics✅ **Scales** to high-dimensional spaces**Three Approaches to Skill Discovery:****1. Diversity-Driven** (unsupervised)- Learn skills that visit diverse states- No task-specific reward needed- Example: DIAYN, VIC**2. Subgoal-Based** (semi-supervised)- Identify bottleneck states as subgoals- Learn skills to reach each subgoal- Example: Option-Critic**3. Task-Driven** (supervised)- Learn skills for specific task families- Requires task distribution- Example: Meta-learning + skills**Comparison:**<table><tr>    <td><b>Approach</b></td>    <td><b>Supervision</b></td>    <td><b>Discovery Mechanism</b></td>    <td><b>Best For</b></td></tr><tr>    <td>Diversity-Driven</td>    <td>None</td>    <td>Maximize state coverage</td>    <td>Unknown task, exploration</td></tr><tr>    <td>Subgoal-Based</td>    <td>Graph/topology</td>    <td>Identify bottlenecks</td>    <td>Structured environments</td></tr><tr>    <td>Task-Driven</td>    <td>Task distribution</td>    <td>Optimize for tasks</td>    <td>Known task family</td></tr></table>**This Tutorial Focus**: We'll implement **DIAYN** (diversity-driven), which:- Requires no task information- Discovers diverse, useful skills- Is widely used in research and practice

<a name='2'></a>## 2 - Setup and Dependencies

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimfrom torch.distributions import Categoricalimport numpy as npimport matplotlib.pyplot as pltfrom matplotlib.patches import Rectanglefrom tqdm import tqdmimport syssys.path.append('..')from utils.test_utils import print_success, print_hint, HintSystemfrom utils.data_utils import set_seedset_seed(42)device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"✅ Device: {device}")print(f"✅ Setup complete!")

<a name='3'></a>## 3 - Theoretical Background<a name='3-1'></a>### 3.1 - Options FrameworkThe **Options Framework** (Sutton et al., 1999) formalizes temporally-extended actions.**Semi-MDP with Options:**An MDP augmented with options: $(S, A, O, P, R, \gamma)$- $S$: State space- $A$: Primitive actions- $O$: Set of options- $P$: Transition dynamics- $R$: Reward function- $\gamma$: Discount factor**Policy over Options:**The agent has a **meta-policy** $\mu(o|s)$ that selects options.**Execution:**1. In state $s_t$, meta-controller selects option $o \sim \mu(s_t)$2. Execute option's policy $a \sim \pi_o(s)$ until termination3. Sample termination $\text{terminate} \sim \beta_o(s)$4. If terminated, return to step 1**Value Functions:****Option value** (Q-value for options):$$Q^{\mu}(s, o) = \mathbb{E} \left[ \sum_{t=0}^{\infty} \gamma^t r_t \mid s_0 = s, o_0 = o, \pi = \pi_o, \mu \right]$$**State value**:$$V^{\mu}(s) = \sum_{o} \mu(o|s) Q^{\mu}(s, o)$$**Bellman Equation for Options:**$$Q^{\mu}(s, o) = r(s, a) + \gamma \left[ (1 - \beta(s')) Q^{\mu}(s', o) + \beta(s') V^{\mu}(s') \right]$$where $s'$ is next state and $a \sim \pi_o(s)$.**Key Insight**: Options provide **temporal abstraction** - one choice leads to multiple actions.**Learning with Options:**Can use standard RL algorithms:- Q-learning over options (SMDP Q-learning)- Actor-critic with options- Policy gradient with options**Intra-option Learning**: Update option values even when not executing them (speeds learning).

<a name='3-2'></a>### 3.2 - DIAYN: Diversity is All You Need**DIAYN** (Eysenbach et al., 2018) discovers skills by **maximizing diversity** without any task reward.**Core Idea:**Learn skills $z$ such that:1. **Skills are distinguishable** from states they visit2. **Skills visit diverse states****Objective:**Maximize mutual information between skills and states:$$\max_{\theta} I(S; Z) = H(Z) - H(Z|S)$$where:- $Z$: Skill (latent variable)- $S$: State- $I(S; Z)$: Mutual information**Intuition:**- $H(Z)$: Entropy of skills → Use all skills equally- $H(Z|S)$: Conditional entropy → States reveal which skill was used**Maximize $I(S;Z)$** means: "If I see a state, I should be able to guess which skill generated it."**Practical Implementation:**DIAYN uses a **discriminator** $q(z|s)$ to predict skill from state.**Intrinsic Reward:**$$r_{\text{DIAYN}}(s, z) = \log q(z|s) - \log p(z)$$- $\log q(z|s)$: How well does state $s$ identify skill $z$?- $\log p(z)$: Prior over skills (usually uniform)If skills are uniform: $r_{\text{DIAYN}}(s, z) = \log q(z|s) + \text{const}$**Algorithm:**```Initialize:  Skill policy π_θ(a|s,z)  Discriminator q_φ(z|s)for episode = 1 to N:  Sample skill z ~ Uniform({1,...,K})  for t = 1 to T:    Execute: a ~ π_θ(a|s,z)    Observe: s'    Reward: r = log q_φ(z|s')    Train discriminator: maximize log q_φ(z|s')    Train policy: maximize r using RL (SAC, PPO, etc.)```**Why This Works:**1. Discriminator tries to identify skill from state2. Policy tries to visit states that reveal its skill3. Result: Different skills visit different states → diversity!**Advantages:**✅ No task reward needed✅ Discovers diverse, useful behaviors✅ Skills can be reused for downstream tasks**Limitations:**⚠️ May learn "trivial" skills (e.g., random movement)⚠️ Number of skills $K$ is hyperparameter⚠️ No guarantee skills are useful for any task

<a name='3-3'></a>### 3.3 - Option-Critic Architecture**Option-Critic** (Bacon et al., 2017) learns options **end-to-end** with policy gradient.**Architecture Components:**1. **Option policy**: $\pi_{\theta}(a|s,o)$ - actions given skill2. **Termination function**: $\beta_{\theta}(s, o)$ - when to stop skill3. **Option value**: $Q_{\omega}(s, o)$ - value of each skill4. **Meta-policy**: $\mu(o|s) \propto \exp(Q_{\omega}(s, o))$ or learned**Gradient for Option Policy:**$$\nabla_{\theta} J = \mathbb{E} \left[ \sum_t \nabla_{\theta} \log \pi_{\theta}(a_t | s_t, o_t) A^o(s_t, o_t, a_t) \right]$$where $A^o$ is the advantage for option $o$.**Gradient for Termination:**$$\nabla_{\theta} J_{\beta} = \mathbb{E} \left[ \beta_{\theta}(s, o) \left( V(s) - Q(s, o) + A^o(s, o) \right) \right]$$**Intuition**: Terminate if continuing is worse than average.**Comparison: DIAYN vs Option-Critic**| Aspect | DIAYN | Option-Critic ||--------|-------|---------------|| Objective | Maximize diversity | Maximize task reward || Termination | Fixed time / manual | Learned || Task reward | Not used | Required || Discovery | Unsupervised | Supervised || Best for | Pre-training, exploration | Task-specific skills |

<a name='3-4'></a>### 3.4 - Successor Features**Successor Features** (Barreto et al., 2017) enable **zero-shot transfer** of skills.**Key Idea:**Decompose value function:$$Q(s, a) = \phi(s, a)^T w$$where:- $\phi(s, a)$: **Successor features** - expected discounted features- $w$: **Reward weights** - task-specific**Successor Feature:**$$\psi^{\pi}(s, a) = \mathbb{E}^{\pi} \left[ \sum_{t=0}^{\infty} \gamma^t \phi(s_t, a_t) \mid s_0=s, a_0=a \right]$$**Intuition**: "What features will I see if I take action $a$ then follow $\pi$?"**Transfer:**If two tasks share features $\phi$ but have different rewards $w_1, w_2$:$$Q_1(s,a) = \psi(s,a)^T w_1$$$$Q_2(s,a) = \psi(s,a)^T w_2$$**Learn** $\psi$ on task 1, **transfer** to task 2 by just changing $w$!**Example:**```Robot manipulation:  Features: {gripper open, object position, arm angle, ...}Task 1: Pick up red cube  w_1 = [0, 1, 0, 0, ...] # reward for object at targetTask 2: Pick up blue sphere  w_2 = [0, 0, 1, 0, ...] # same features, different targetTransfer: Use same ψ, just change w!```**Skills + Successor Features:**Learn a **library of skills**, each with successor features $\psi_i$.For new task with reward $w$:- Compute: $Q_i = \psi_i^T w$ for each skill $i$- Use best skill: $i^* = \arg\max_i \psi_i(s, a)^T w$**Zero-shot transfer** without any additional training!

<a name='ex-1'></a>## 4 - Exercise 1: Grid World EnvironmentImplement a grid world environment for skill discovery experiments.

In [ ]:
class GridWorldEnv:    """    Simple grid world for skill discovery.    Agent can move in 4 directions.    Goal: Learn diverse skills (e.g., go to corners, explore edges, etc.)    """    def __init__(self, size=7):        self.size = size        self.n_actions = 4  # up, right, down, left        self.n_states = size * size        self.reset()    def reset(self):        """Reset to center of grid."""        self.pos = np.array([self.size // 2, self.size // 2])        return self.get_state()    def get_state(self):        """        Return state as one-hot encoded position.        Returns:            state: [size*size] one-hot vector        """        # TODO: Convert 2D position to one-hot state vector        # Hint: Use np.zeros and set appropriate index to 1        state = np.zeros(self.n_states)        idx = self.pos[0] * self.size + self.pos[1]        state[idx] = 1.0        return state    def get_position(self):        """Return (x, y) position."""        return tuple(self.pos)    def step(self, action):        """        Execute action.        Args:            action: int in {0,1,2,3} for {up,right,down,left}        Returns:            next_state: [n_states] one-hot vector            reward: 0 (no task reward for skill discovery)            done: False (continuous task)            info: dict        """        # TODO: Implement grid world dynamics        # Actions: 0=up, 1=right, 2=down, 3=left        # Hint: Update self.pos, clip to bounds [0, size-1]        # Define movement        moves = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}        move = moves[action]        # Update position        new_pos = self.pos + np.array(move)        # Clip to grid bounds        new_pos = np.clip(new_pos, 0, self.size - 1)        self.pos = new_pos        # Return state, reward, done, info        return self.get_state(), 0.0, False, {}# Test environmentenv = GridWorldEnv(size=7)state = env.reset()print("🗺️  Grid World Environment")print(f"  Grid size: {env.size}x{env.size}")print(f"  State dim: {env.n_states}")print(f"  Actions: {env.n_actions}")print(f"  Initial position: {env.get_position()}")# Test movementprint("\n  Testing movements:")for action, name in enumerate(['UP', 'RIGHT', 'DOWN', 'LEFT']):    env.reset()    next_state, _, _, _ = env.step(action)    print(f"    {name}: {env.get_position()}")# Hintshints_env = HintSystem([    "State is one-hot: state[row * size + col] = 1",    "Movements: {0: (-1,0), 1: (0,1), 2: (1,0), 3: (0,-1)}",    "Use np.clip(pos, 0, size-1) to stay in bounds",])hints_env.show_hint()

<a name='ex-2'></a>## 5 - Exercise 2: Skill DiscriminatorImplement the discriminator for DIAYN.

In [ ]:
class SkillDiscriminator(nn.Module):    """    Discriminator q(z|s) for DIAYN.    Predicts which skill generated a state.    """    def __init__(self, state_dim, n_skills, hidden_dim=128):        super(SkillDiscriminator, self).__init__()        # TODO: Define network to predict skill from state        # Input: state [batch, state_dim]        # Output: skill probabilities [batch, n_skills]        self.net = nn.Sequential(            nn.Linear(state_dim, hidden_dim),            nn.ReLU(),            nn.Linear(hidden_dim, hidden_dim),            nn.ReLU(),            nn.Linear(hidden_dim, n_skills)        )    def forward(self, state):        """        Predict skill from state.        Args:            state: [batch, state_dim]        Returns:            skill_logits: [batch, n_skills]        """        # TODO: Forward pass through network        return self.net(state)    def predict_log_prob(self, state, skill):        """        Compute log q(z|s).        Args:            state: [batch, state_dim]            skill: [batch] - skill indices        Returns:            log_prob: [batch] - log probabilities        """        # TODO: Compute log probability of skill given state        # Hint: Use F.log_softmax and gather        logits = self.forward(state)        log_probs = F.log_softmax(logits, dim=-1)        # Gather log prob for actual skill        skill_log_probs = log_probs.gather(1, skill.unsqueeze(-1)).squeeze(-1)        return skill_log_probs# Test discriminatorstate_dim = 49  # 7x7 gridn_skills = 4discriminator = SkillDiscriminator(state_dim, n_skills).to(device)# Test forward passbatch_size = 16test_states = torch.randn(batch_size, state_dim).to(device)test_skills = torch.randint(0, n_skills, (batch_size,)).to(device)logits = discriminator(test_states)log_probs = discriminator.predict_log_prob(test_states, test_skills)print("🎯 Skill Discriminator")print(f"  Input shape: {test_states.shape}")print(f"  Output logits shape: {logits.shape}")print(f"  Log probs shape: {log_probs.shape}")print(f"  Log prob range: [{log_probs.min():.3f}, {log_probs.max():.3f}]")# Check that probabilities sum to 1probs = F.softmax(logits, dim=-1)print(f"  Probabilities sum: {probs.sum(dim=-1).mean():.4f} (should be 1.0)")# Hintshints_disc = HintSystem([    "Use nn.Sequential for simple feed-forward network",    "Forward: just return self.net(state)",    "Log prob: F.log_softmax(logits, -1), then .gather(1, skill.unsqueeze(-1))",])hints_disc.show_hint()

<a name='ex-3'></a>## 6 - Exercise 3: Skill PolicyImplement a policy conditioned on skill.

In [ ]:
class SkillPolicy(nn.Module):    """    Policy π(a|s,z) conditioned on skill z.    """    def __init__(self, state_dim, n_skills, n_actions, hidden_dim=128):        super(SkillPolicy, self).__init__()        self.n_skills = n_skills        self.n_actions = n_actions        # TODO: Define network        # Input: state + one-hot skill        # Output: action logits        self.net = nn.Sequential(            nn.Linear(state_dim + n_skills, hidden_dim),            nn.ReLU(),            nn.Linear(hidden_dim, hidden_dim),            nn.ReLU(),            nn.Linear(hidden_dim, n_actions)        )    def forward(self, state, skill):        """        Compute action probabilities.        Args:            state: [batch, state_dim]            skill: [batch] - skill indices        Returns:            action_probs: [batch, n_actions]        """        # TODO: 1) Convert skill to one-hot        #       2) Concatenate with state        #       3) Pass through network        #       4) Apply softmax        # One-hot encode skill        skill_onehot = F.one_hot(skill, self.n_skills).float()        # Concatenate state and skill        x = torch.cat([state, skill_onehot], dim=-1)        # Forward pass        logits = self.net(x)        action_probs = F.softmax(logits, dim=-1)        return action_probs    def select_action(self, state, skill):        """        Sample action from policy.        Returns:            action: int            log_prob: Tensor        """        probs = self.forward(state, skill)        dist = Categorical(probs)        action = dist.sample()        log_prob = dist.log_prob(action)        return action, log_prob# Test policypolicy = SkillPolicy(state_dim=49, n_skills=4, n_actions=4).to(device)# Test with single statestate = torch.FloatTensor(env.get_state()).unsqueeze(0).to(device)skill = torch.LongTensor([0]).to(device)action_probs = policy(state, skill)action, log_prob = policy.select_action(state, skill)print("🎮 Skill Policy")print(f"  State shape: {state.shape}")print(f"  Action probs: {action_probs[0].detach().cpu().numpy()}")print(f"  Selected action: {action.item()}")print(f"  Log prob: {log_prob.item():.3f}")# Test different skills give different actionsprint("\n  Testing skill conditioning:")for z in range(4):    skill_tensor = torch.LongTensor([z]).to(device)    probs = policy(state, skill_tensor)[0].detach().cpu().numpy()    print(f"    Skill {z}: {probs}")# Hintshints_policy = HintSystem([    "One-hot: F.one_hot(skill, n_skills).float()",    "Concatenate: torch.cat([state, skill_onehot], dim=-1)",    "Softmax: F.softmax(logits, dim=-1)",])hints_policy.show_hint()

<a name='ex-4'></a>## 7 - Exercise 4: DIAYN Training LoopImplement the complete DIAYN training loop.

In [ ]:
def train_diayn(    env,    n_skills=4,    n_episodes=500,    episode_length=50,    lr_policy=3e-4,    lr_discriminator=3e-4,    gamma=0.99):    """    Train skills using DIAYN algorithm.    Returns:        policy: trained policy        discriminator: trained discriminator        history: training metrics    """    state_dim = env.n_states    n_actions = env.n_actions    # Initialize networks    policy = SkillPolicy(state_dim, n_skills, n_actions).to(device)    discriminator = SkillDiscriminator(state_dim, n_skills).to(device)    # Optimizers    policy_optimizer = optim.Adam(policy.parameters(), lr=lr_policy)    disc_optimizer = optim.Adam(discriminator.parameters(), lr=lr_discriminator)    # Logging    history = {        'disc_loss': [],        'policy_reward': [],        'disc_accuracy': []    }    pbar = tqdm(range(n_episodes), desc="Training DIAYN")    for episode in pbar:        # Sample skill uniformly        skill = torch.randint(0, n_skills, (1,)).item()        skill_tensor = torch.LongTensor([skill]).to(device)        # Collect episode        state = env.reset()        states, actions, log_probs, rewards = [], [], [], []        for step in range(episode_length):            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)            # Select action            action, log_prob = policy.select_action(state_tensor, skill_tensor)            # Step environment            next_state, _, done, _ = env.step(action.item())            # Compute DIAYN intrinsic reward            next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0).to(device)            with torch.no_grad():                reward = discriminator.predict_log_prob(next_state_tensor, skill_tensor)                reward = reward.item()            # Store transition            states.append(state_tensor)            actions.append(action)            log_probs.append(log_prob)            rewards.append(reward)            state = next_state        # Convert to tensors        states = torch.cat(states)        log_probs = torch.stack(log_probs)        rewards = torch.FloatTensor(rewards).to(device)        # Compute returns        returns = []        R = 0        for r in reversed(rewards.tolist()):            R = r + gamma * R            returns.insert(0, R)        returns = torch.FloatTensor(returns).to(device)        returns = (returns - returns.mean()) / (returns.std() + 1e-8)        # Update policy (REINFORCE)        policy_loss = -(log_probs * returns).mean()        policy_optimizer.zero_grad()        policy_loss.backward()        policy_optimizer.step()        # Update discriminator        skills_batch = skill_tensor.expand(states.shape[0])        disc_logits = discriminator(states)        disc_loss = F.cross_entropy(disc_logits, skills_batch)        disc_optimizer.zero_grad()        disc_loss.backward()        disc_optimizer.step()        # Compute accuracy        with torch.no_grad():            pred_skills = disc_logits.argmax(dim=-1)            accuracy = (pred_skills == skills_batch).float().mean().item()        # Logging        history['disc_loss'].append(disc_loss.item())        history['policy_reward'].append(rewards.mean().item())        history['disc_accuracy'].append(accuracy)        pbar.set_postfix({            'disc_loss': f'{disc_loss.item():.3f}',            'reward': f'{rewards.mean().item():.3f}',            'acc': f'{accuracy:.3f}'        })    return policy, discriminator, historyprint("🚀 Starting DIAYN Training...")print("  Learning diverse skills without task reward\n")env = GridWorldEnv(size=7)policy, discriminator, history = train_diayn(    env,    n_skills=4,    n_episodes=500,    episode_length=50)print(f"\n✅ Training Complete!")print(f"  Final disc accuracy: {history['disc_accuracy'][-1]:.3f}")print(f"  Final reward: {history['policy_reward'][-1]:.3f}")

<a name='9'></a>## 9 - Visualization: Skill BehaviorsVisualize what each skill learned.

In [ ]:
# Plot training curvesfig, axes = plt.subplots(1, 3, figsize=(16, 4))# Discriminator lossaxes[0].plot(history['disc_loss'], color='steelblue', alpha=0.7)axes[0].set_xlabel('Episode', fontsize=12)axes[0].set_ylabel('Discriminator Loss', fontsize=12)axes[0].set_title('Discriminator Training', fontsize=14, fontweight='bold')axes[0].grid(True, alpha=0.3)# Rewardaxes[1].plot(history['policy_reward'], color='forestgreen', alpha=0.7)axes[1].set_xlabel('Episode', fontsize=12)axes[1].set_ylabel('Avg Intrinsic Reward', fontsize=12)axes[1].set_title('Policy Reward', fontsize=14, fontweight='bold')axes[1].grid(True, alpha=0.3)# Accuracyaxes[2].plot(history['disc_accuracy'], color='coral', alpha=0.7)axes[2].axhline(y=0.25, color='red', linestyle='--', alpha=0.5, label='Random')axes[2].set_xlabel('Episode', fontsize=12)axes[2].set_ylabel('Discriminator Accuracy', fontsize=12)axes[2].set_title('Skill Distinguishability', fontsize=14, fontweight='bold')axes[2].legend()axes[2].grid(True, alpha=0.3)plt.tight_layout()plt.show()# Visualize skill behaviorsfig, axes = plt.subplots(2, 2, figsize=(12, 12))for skill_idx in range(4):    ax = axes[skill_idx // 2, skill_idx % 2]    # Collect trajectory with this skill    env.reset()    trajectory = [env.get_position()]    visit_counts = np.zeros((env.size, env.size))    state = env.get_state()    skill = torch.LongTensor([skill_idx]).to(device)    for _ in range(100):        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)        with torch.no_grad():            action, _ = policy.select_action(state_tensor, skill)        state, _, _, _ = env.step(action.item())        pos = env.get_position()        trajectory.append(pos)        visit_counts[pos[0], pos[1]] += 1    # Plot heatmap    im = ax.imshow(visit_counts, cmap='YlOrRd', origin='upper')    ax.set_title(f'Skill {skill_idx} State Visitation', fontsize=14, fontweight='bold')    # Add colorbar    plt.colorbar(im, ax=ax, fraction=0.046)    # Mark start    start_pos = trajectory[0]    ax.plot(start_pos[1], start_pos[0], 'bo', markersize=15, label='Start')    # Add grid    ax.set_xticks(np.arange(-.5, env.size, 1), minor=True)    ax.set_yticks(np.arange(-.5, env.size, 1), minor=True)    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)    ax.legend()plt.suptitle('DIAYN: Learned Diverse Skills', fontsize=16, fontweight='bold')plt.tight_layout()plt.show()print("📊 Observations:")print("  • Each skill visits different regions of the state space")print("  • Skills emerge without any task-specific reward")print("  • High discriminator accuracy → skills are distinguishable")print("  • These skills can now be reused for downstream tasks!")

<a name='12'></a>## 12 - Real-World Applications of Skill DiscoverySkill discovery has shown significant impact in real-world domains:### 1. **Robotics** 🤖**Manipulation:**- **Problem**: Learning to manipulate hundreds of different objects- **Skill Discovery Solution**: Learn skills like {grasp, push, pull, rotate}- **Result**: Transfer to new objects with 10x fewer demonstrations**Example: UC Berkeley Robot Learning Lab**- Pre-train with DIAYN: 100,000 random interactions- Discover skills: reach, grasp, place, push- Fine-tune on tasks: 1,000 interactions per task (vs 50,000 from scratch)- **50x speedup****Locomotion:**- Skills: {walk forward, turn left, turn right, climb, crawl}- Application: Quadruped robots adapting to terrain- Discovered skills transfer across different terrains### 2. **Video Games** 🎮**Montezuma's Revenge:**- Notoriously hard exploration problem- DIAYN discovers skills: climb ladder, avoid skull, collect key- Performance: 0 score (random) → 2,500 score (DIAYN skills)**Minecraft:**- Skills discovered: mine, craft, build, explore- Enables hierarchical planning for complex goals- "Build house" = compose {gather wood, craft planks, place blocks}### 3. **Autonomous Driving** 🚗**Skills for Different Scenarios:**- Lane keeping- Lane changing- Merging- Parking- Emergency braking**Advantage:** Can mix and match skills for new scenarios- New city? Reuse lane keeping + adapt merging to local traffic patterns### 4. **Industrial Automation** 🏭**Assembly Line:**- Skills: {pick component, align, insert, fasten, inspect}- New product? Compose existing skills in new sequence- Reduces reprogramming time from weeks to hours**Warehouse Robots:**- Skills: {navigate to shelf, scan barcode, pick item, place in bin}- Handles new inventory without retraining### 5. **Healthcare and Surgery** 🏥**Robotic Surgery:**- Skills: {suture, cut, cauterize, grasp tissue}- Different procedures compose these basic skills- Surgeons can teach new procedures by specifying skill sequences### Real Impact Numbers:| Domain | Task | Without Skills | With Skills | Improvement ||--------|------|----------------|-------------|-------------|| Robot manipulation | Pick new object | 50,000 samples | 1,000 samples | 50x || Montezuma's Revenge | Game score | 0 | 2,500 | ∞ || Quadruped locomotion | Adapt to terrain | 200,000 steps | 10,000 steps | 20x || Warehouse robot | New product type | 20 hours training | 1 hour | 20x |### Success Stories:**1. Google Brain - Robotic Grasping (2019)**- Discovered 12 manipulation skills- Transferred to 50+ different objects- Grasping success rate: 96% (vs 78% without skills)**2. DeepMind - Capture the Flag (2019)**- Discovered movement and strategy skills- Defeated human players- Skills transferred to new map layouts**3. OpenAI - Hide and Seek (2019)**- Emerged skills: box surfing, ramp use, tool manipulation- Completely unexpected behaviors- Demonstrated potential of emergent complexity### Challenges in Deployment:⚠️ **Safety**: Ensuring skills don't include dangerous behaviors⚠️ **Interpretability**: Understanding what each skill does⚠️ **Robustness**: Skills must work reliably in real world⚠️ **Coverage**: Need enough skills to handle all scenarios

<a name='14'></a>## 14 - Summary and Conclusions<font color='blue'>**What you should remember:**✅ **Skills are temporally-extended actions** that provide abstraction and transfer✅ **Options Framework**: Formal definition with initiation, policy, termination✅ **DIAYN**: Learn diverse skills by maximizing $I(S;Z)$ without task reward✅ **Skill Discovery**: Enables exploration, transfer, and compositionality✅ **Applications**: Robotics, games, automation benefit from reusable skills✅ **Discriminator**: Learns to identify skills from states visited✅ **Intrinsic Reward**: $r = \log q(z|s)$ encourages distinguishable behaviors</font>### Key Takeaways:**1. The Compositionality Principle:**- Complex behaviors = compositions of simple skills- Humans do this naturally, AI can learn it too**2. Discovery vs Design:**- Manual skill design: requires expertise, limited- Automatic discovery: scalable, finds unexpected skills**3. Diversity is Key:**- DIAYN maximizes mutual information $I(S;Z)$- Diverse skills are more likely to be useful**4. Transfer is the Goal:**- Skills learned once, reused many times- 10-50x speedup on new tasks### Comparison of Methods:<table><tr>    <td><b>Method</b></td>    <td><b>Supervision</b></td>    <td><b>Objective</b></td>    <td><b>Best For</b></td></tr><tr>    <td>DIAYN</td>    <td>None</td>    <td>Maximize diversity</td>    <td>Exploration, pre-training</td></tr><tr>    <td>Option-Critic</td>    <td>Task reward</td>    <td>Maximize return</td>    <td>Task-specific skills</td></tr><tr>    <td>Successor Features</td>    <td>Feature design</td>    <td>Transfer Q-values</td>    <td>Related tasks</td></tr><tr>    <td>Meta-Learning + Skills</td>    <td>Task distribution</td>    <td>Fast adaptation</td>    <td>Task families</td></tr></table>### Practical Guidelines:**When to use skill discovery:**✅ Multiple related tasks✅ Need for transfer and reuse✅ Complex behaviors require composition✅ Exploration is challenging**When NOT to use:**❌ Single simple task❌ Tasks are completely unrelated❌ Flat policy works well**Implementation Tips:**1. **Start with DIAYN** for simplicity2. **Choose K carefully**: Too few → not enough diversity; too many → hard to distinguish3. **Monitor discriminator accuracy**: Should be >> 1/K4. **Visualize skills**: Understand what each one does5. **Fine-tune for tasks**: Skills are starting point, not final solution### 📚 Essential References:1. **Options Framework**: [Sutton et al., 1999](https://people.cs.umass.edu/~barto/courses/cs687/Sutton-Precup-Singh-AIJ99.pdf)2. **DIAYN**: [Eysenbach et al., 2018](https://arxiv.org/abs/1802.06070)3. **Option-Critic**: [Bacon et al., 2017](https://arxiv.org/abs/1609.05140)4. **Successor Features**: [Barreto et al., 2017](https://arxiv.org/abs/1606.05312)5. **VIC** (improved DIAYN): [Gregor et al., 2016](https://arxiv.org/abs/1611.07507)### 🚀 Next Steps:1. Implement DIAYN on continuous control tasks (Mujoco)2. Try Option-Critic with task rewards3. Explore Successor Features for transfer4. Combine with Meta-Learning for fast skill acquisition5. Apply to your own domain!---## 🎉 Congratulations!You now understand **Skill Discovery**, a powerful paradigm for building compositional, reusable AI!**You've learned**:- Options framework and temporal abstraction- DIAYN algorithm for unsupervised discovery- How discriminators enable diversity- Real-world applications and impact**You're ready to**:- Implement skill discovery systems- Design hierarchical RL agents- Apply to robotics, games, automation- Research new discovery methods!**Next Tutorial**: Tutorial 08 explores generalization to out-of-distribution scenarios! 🌐